In [1]:
%load_ext autoreload
%autoreload 2

#### This scrapt is used for match station, remove nan values, and match unit

In [2]:
from crmprtd import Row
import logging
import os
import pickle
import pandas as pd
import numpy as np
from natsort import natsorted
from natsort import natsort_keygen
from pprint import pprint
from collections import Counter
from collections import defaultdict

import re
from rapidfuzz import fuzz
from crmprtd import infer
from fern_func import *
from sql_func import *

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)


In [3]:
# --- Main workflow ---
# --- read data ---
meta_path = '/fern_data/FERNNorth2024_VF/20241125 MeteorologicalNetworks-FERN-VF-shared.xlsx'
data_path = '/fern_data/FERNNorth2024_VF/WxData24'

df_station = pd.read_excel(meta_path)
df_station = df_station.sort_values(by='station_name', key=natsort_keygen())

csv_files = [f for f in os.listdir(data_path) if f.endswith('.csv')]
# Sort in natural order
csv_files = natsorted(csv_files)

# --- output folder ---
output_folder = '/workspaces/crmprtd/fern/all_stations_insert/rows_output/'
os.makedirs(output_folder, exist_ok=True)


In [4]:
df_station

,lat,lon,ECO Province,native_id,Eco Province Octile,station_name,Station_name uppercase,Agency-Subnetwork,elev,ClimateDesignated,...,Humidity,Precipitation,SnowDepth,SnowWater,Winds,Pressure,Radiation,Soil,Non Shef Code parameter,Water
0,59.574000,-133.687000,NORTHERN BOREAL MOUNTAINS,AtlSch,1.0,Atlin School,ATLIN SCHOOL,FOR/FERN,705,NaN,...,XR,PP(TBRG),NaN,NaN,USUW,BP,RW,NaN,NaN,NaN
1,54.510444,-126.614341,CENTRAL INTERIOR,Barren,4.0,BarrenWx,BARRENWX,FOR/FERN,1051,NaN,...,XR,PP(TBRG),NaN,NaN,USUW(3.5m),BP,RW,MS;TB;TS,NaN,NaN
2,55.078885,-120.352171,BOREAL PLAINS,Blackhawk,8.0,BlackhawkWx,BLACKHAWKWX,FOR/FERN,962,NaN,...,XR,PP(TBRG),SD,NaN,USUW(3.5m),BP,RW,MS;TB;TS,NaN,NaN
3,55.108173,-127.374740,COAST AND MOUNTAINS,Boulder Creek,3.0,BoulderWx,BOULDERWX,FOR/FERN,385,NaN,...,XR,PP(TBRG),NaN,NaN,USUW(3.5m),BP,RW,MS;TB,NaN,NaN
4,53.902111,-122.015389,SUB-BOREAL INTERIOR,BowronPit,1.0,BowronPit,BOWRONPIT,FOR/FERN,722,NaN,...,XR,PP(TBRG),SD,NaN,USUW(3.5m),BP,RW,MS;TB,NaN,NaN
5,53.772183,-122.720729,SUB-BOREAL INTERIOR,PGTIS1,1.0,BulkleyWx,BULKLEYWX,FOR/FERN,594,NaN,...,XR,PP(TBRG),NaN,NaN,USUW(3.5m),BP,RW,MS;TB,NaN,NaN
11,53.753775,-122.718864,SUB-BOREAL INTERIOR,PGTIS3,1.0,CPFWx,CPFWX,FOR/FERN,593,NaN,...,XR,PP(TBRG),NaN,NaN,USUW(3.5m),BP,RW,NaN,NaN,NaN
6,52.709820,-119.191260,SOUTHERN INTERIOR MOUNTAINS,Canoe Mountain Stn,8.0,Canoe Mountain Stn,CANOE MOUNTAIN STN,FOR/FERN,2210,NaN,...,XR,PP(TBRG),NaN,NaN,USUW(3.5m),BP,RW,TS,NaN,NaN
7,58.357180,-129.546370,NORTHERN BOREAL MOUNTAINS,CaribouPass,8.0,Caribou Pass,CARIBOU PASS,FOR/FERN,1744,NaN,...,XR,PP(TBRG),NaN,NaN,USUW(3.5m),BP,RW,TS,NaN,NaN
8,54.883903,-126.623533,SUB-BOREAL INTERIOR,Chapman,2.0,ChapmanWx,CHAPMANWX,FOR/FERN,848,NaN,...,XR,PP(TBRG),NaN,NaN,USUW(3.5m),BP,RW,MS;TB;TS,NaN,NaN


In [5]:
import os
import pandas as pd
import logging
from collections import Counter, defaultdict

# --- Setup logger to write to a file ---
log_path = '/workspaces/crmprtd/fern/all_stations_insert/'
log_file = "fern_all_stations_raw_summary.log"
logger = logging.getLogger("station_logger")
logger.setLevel(logging.INFO)

if logger.hasHandlers():
    logger.handlers.clear()

fh = logging.FileHandler(os.path.join(log_path, log_file), mode="w", encoding="utf-8")
fh.setLevel(logging.INFO)
formatter = logging.Formatter("%(message)s")
fh.setFormatter(formatter)
logger.addHandler(fh)

# --- Define column widths for pretty logging ---
col_widths = {
    "variable": 25,
    "unit": 12,
    "count": 8,
    "start_time": 22,
    "end_time": 22
}

# --- Loop through stations and generate raw summaries ---
for station_i, csv_file in enumerate(csv_files[0:10]):
    station_file_name = os.path.splitext(csv_file)[0]
    df_data = pd.read_csv(os.path.join(data_path, csv_file), encoding='latin1', low_memory=False, nrows=100)
    
    station_name = df_station.iloc[station_i]['station_name']
    
    if names_match(station_name, station_file_name):
        logger.info(f"\n{'='*100}")
        logger.info(f"📍 Station: {station_name} (File: {station_file_name})")
        logger.info(f"{'='*100}")

        rows, native_id = create_rows_from_dataframe(df_station, df_data, station_i)
        rows_clean = [r for r in rows if not pd.isna(r.val)]
        
        # Count variables
        var_counts = Counter(r.variable_name for r in rows_clean if hasattr(r, "variable_name"))
        total_var_counts = sum(var_counts.values())
        variable_unit_fern = [(r.variable_name, r.unit) for r in rows_clean]
        unique_variable_unit_fern = list(set(variable_unit_fern))
        var_to_unit = {var: unit for var, unit in unique_variable_unit_fern}

        # Collect times per variable
        var_times = defaultdict(list)
        for r in rows_clean:
            if hasattr(r, "variable_name") and hasattr(r, "time"):
                var_times[r.variable_name].append(r.time)

        # --- Log header ---
        logger.info(f"{'Variable':{col_widths['variable']}} "
                    f"{'Unit':{col_widths['unit']}} "
                    f"{'Count':{col_widths['count']}} "
                    f"{'Start Time':{col_widths['start_time']}} "
                    f"{'End Time':{col_widths['end_time']}}")
        logger.info("-" * sum(col_widths.values()))

        # --- Log each variable ---
        for var, count in var_counts.items():
            unit = var_to_unit.get(var, 'N/A')
            times = var_times.get(var, [])
            start_time = min(times) if times else "N/A"
            end_time = max(times) if times else "N/A"
            logger.info(f"{var:{col_widths['variable']}} "
                        f"{unit:{col_widths['unit']}} "
                        f"{count:{col_widths['count']}} "
                        f"{str(start_time):{col_widths['start_time']}} "
                        f"{str(end_time):{col_widths['end_time']}}")

        # --- Totals ---
        total_var_types = len(var_counts)
        logger.info("-" * sum(col_widths.values()))
        logger.info(f"{'TOTAL':{col_widths['variable']}} "
                    f"{'':{col_widths['unit']}} "
                    f"{total_var_counts:{col_widths['count']}} "
                    f"{'':{col_widths['start_time']}} "
                    f"{'':{col_widths['end_time']}}")
        logger.info(f"Total variable types : {total_var_types}")
        logger.info(f"Total variable counts: {total_var_counts}")
        logger.info("\n")  # spacing between stations

logger.info("✅ All stations summarized successfully.")

2025-11-03 21:27:54 - INFO - 
2025-11-03 21:27:54 - INFO - 📍 Station: Atlin School (File: Atlin school)
2025-11-03 21:27:54 - INFO - ====================================================================================================
2025-11-03 21:27:54 - INFO - Variable                  Unit         Count    Start Time             End Time              
2025-11-03 21:27:54 - INFO - -----------------------------------------------------------------------------------------
2025-11-03 21:27:54 - INFO - Rain                      mm                 99 2010-08-20 20:00:00    2010-08-24 22:00:00   
2025-11-03 21:27:54 - INFO - Air Temp                  °C                100 2010-08-20 19:00:00    2010-08-24 22:00:00   
2025-11-03 21:27:54 - INFO - Wind Speed                m/s               100 2010-08-20 19:00:00    2010-08-24 22:00:00   
2025-11-03 21:27:54 - INFO - Gust Speed                m/s               100 2010-08-20 19:00:00    2010-08-24 22:00:00   
2025-11-03 21:27:54 - INFO - Win

### Check the DB - variable name, unit, start_time, end_time for each station

In [7]:
import sqlalchemy as sa
import pandas as pd
import logging

# --- Setup database connection ---
engine = sa.create_engine("postgresql://tongli1997@pg01.pcic.uvic.ca:5432/crmp")

# --- Setup logger ---
log_path = '/workspaces/crmprtd/fern/all_stations_insert/'
log_file = "fern_db_summary.log"

logger = logging.getLogger("station_logger")
logger.setLevel(logging.INFO)

if logger.hasHandlers():
    logger.handlers.clear()

fh = logging.FileHandler(log_path + log_file, mode="w", encoding="utf-8")
fh.setLevel(logging.INFO)
formatter = logging.Formatter("%(message)s")
fh.setFormatter(formatter)
logger.addHandler(fh)

# --- Column widths for formatting ---
col_widths = {
    "station_name": 25,
    "net_var_name": 25,
    "unit": 12,
    "earliest_time": 22,
    "latest_time": 22,
    "num_records": 10
}

# --- Manual corrections dictionary ---
# Key = original station, Value = (corrected_name, exact_match)
manual_corrections = {
    'Atlin School': ('Atlin', True),
    'BowronPit': ('Bowron Pit', True),
    'ChiefLakeWx': ('CHIEF LAKE', True),
    'MacJxnWx': ('Mackenzie Jxn', True),
    'BoulderWx': ('BOULDER CREEK', True),
    'Dennis': ('Dennis', True),
    'DunsterWx': ('DUNSTER', True)
}

# --- List to store stations still with no data ---
stations_no_data = []

# --- Function to query and log a station ---
def query_and_log(station_pattern, display_name=None, exact_match=False):
    """
    Query a station and log its data.
    
    :param station_pattern: Pattern to search in the DB
    :param display_name: Name to display in the log (optional)
    :param exact_match: If True, use exact match (=), else use ILIKE
    :return: True if data found, None if empty
    """
    display_name = display_name or station_pattern

    if exact_match:
        where_clause = "m.station_name = :station_pattern"
        pattern_param = station_pattern  # no wildcards — exact but case-insensitive
    else:
        where_clause = "m.station_name ILIKE :station_pattern"
        pattern_param = f"%{station_pattern}%"

    query_text = f"""
        SELECT
            m.station_name,
            v.net_var_name,
            v.unit,
            MIN(o.obs_time) AS earliest_time,
            MAX(o.obs_time) AS latest_time,
            COUNT(*) AS num_records
        FROM obs_raw AS o
        JOIN meta_history AS m
            ON o.history_id = m.history_id
        JOIN meta_vars AS v
            ON o.vars_id = v.vars_id
        WHERE {where_clause}
        GROUP BY m.station_name, v.net_var_name, v.unit
        ORDER BY v.net_var_name;
    """

    query = sa.text(query_text)

    with engine.connect() as conn:
        df = pd.read_sql(query, conn, params={"station_pattern": pattern_param})

    if df.empty:
        return None

    # --- Log table ---
    logger.info(f"\n{'='*140}")
    logger.info(f"📍 Station: {display_name}")
    logger.info(f"{'='*140}")
    logger.info(f"{'Station':{col_widths['station_name']}} "
                f"{'Variable':{col_widths['net_var_name']}} "
                f"{'Unit':{col_widths['unit']}} "
                f"{'Earliest Time':{col_widths['earliest_time']}} "
                f"{'Latest Time':{col_widths['latest_time']}} "
                f"{'Count':{col_widths['num_records']}}")
    logger.info("-" * sum(col_widths.values()))

    for _, row in df.iterrows():
        logger.info(f"{row['station_name']:{col_widths['station_name']}} "
                    f"{row['net_var_name']:{col_widths['net_var_name']}} "
                    f"{(row['unit'] or 'N/A'):{col_widths['unit']}} "
                    f"{str(row['earliest_time']):{col_widths['earliest_time']}} "
                    f"{str(row['latest_time']):{col_widths['latest_time']}} "
                    f"{row['num_records']:{col_widths['num_records']}}")

    # --- Totals ---
    total_var_types = len(df)
    total_var_counts = df['num_records'].sum()
    logger.info("-" * sum(col_widths.values()))
    logger.info(f"{'TOTAL':{col_widths['station_name']}} "
                f"{'':{col_widths['net_var_name']}} "
                f"{'':{col_widths['unit']}} "
                f"{'':{col_widths['earliest_time']}} "
                f"{'':{col_widths['latest_time']}} "
                f"{total_var_counts:{col_widths['num_records']}}")
    logger.info(f"Number of variable types: {total_var_types}\n")

    return True

# --- Main loop for all stations ---
for station_name in df_station['station_name']:
    # 1. Manual corrections only
    if station_name in manual_corrections:
        corrected_name, exact_match = manual_corrections[station_name]
        match_type = "exact match" if exact_match else "LIKE search"
        logger.info(f"🔄 Using manual correction for '{station_name}': '{corrected_name}' ({match_type})...")
        if query_and_log(corrected_name, display_name=f"{station_name} (manual correction: {corrected_name})", exact_match=exact_match):
            continue

    # 2. Try original name
    if query_and_log(station_name):
        continue

    # 3. Try removing 'Wx' suffix
    if station_name.endswith('Wx'):
        revised_name = station_name[:-2].strip()
        logger.info(f"🔄 No data for '{station_name}', retrying as '{revised_name}' (LIKE search)...")
        if query_and_log(revised_name, display_name=f"{station_name} (retry: {revised_name})"):
            continue

    # 4. Still no data
    logger.info(f"❌ No data found for '{station_name}' after all attempts.\n")
    stations_no_data.append(station_name)

logger.info("✅ All stations summarized successfully.")
logger.info(f"Stations with no data after all attempts: {stations_no_data}")

2025-11-03 21:29:44 - INFO - 🔄 Using manual correction for 'Atlin School': 'Atlin' (exact match)...
2025-11-03 21:29:44 - INFO - 
2025-11-03 21:29:44 - INFO - 📍 Station: Atlin School (manual correction: Atlin)
2025-11-03 21:29:44 - INFO - ============================================================================================================================================
2025-11-03 21:29:44 - INFO - Station                   Variable                  Unit         Earliest Time          Latest Time            Count     
2025-11-03 21:29:44 - INFO - --------------------------------------------------------------------------------------------------------------------
2025-11-03 21:29:44 - INFO - Atlin                     CURRENT_AIR_TEMPERATURE1  celsius      1988-01-02 07:30:00    2001-11-02 07:00:00          4800
2025-11-03 21:29:44 - INFO - Atlin                     HEIGHT_OF_SNOW            cm           1988-01-02 07:30:00    2001-11-02 07:00:00          4730
2025-11-03 21:29:44 -

### Compare variables in fiels and in DB

In [8]:
import os
import pandas as pd
import sqlalchemy as sa
from collections import Counter

# --- Setup database connection ---
engine = sa.create_engine("postgresql://tongli1997@pg01.pcic.uvic.ca:5432/crmp")

# --- Manual corrections dictionary ---
manual_corrections = {
    'Atlin School': 'Atlin',
    'BowronPit': 'Bowron Pit',
    'ChiefLakeWx': 'CHIEF LAKE',
    'MacJxnWx': 'Mackenzie Jxn',
    'BoulderWx': 'Boulder Creek',
    'Dennis': 'Dennis',
    'DunsterWx': 'DUNSTER'
}

def find_matching_station_in_db(engine, station_name):
    """Try manual correction, then original name, then 'Wx'-stripped name."""
    # 1. Manual correction (exact match)
    if station_name in manual_corrections:
        corrected = manual_corrections[station_name]
        query = sa.text("""
            SELECT DISTINCT m.station_name
            FROM meta_history AS m
            JOIN obs_raw AS o ON m.history_id = o.history_id
            WHERE m.station_name = :station_pattern
            LIMIT 1;
        """)
        with engine.connect() as conn:
            res = conn.execute(query, {"station_pattern": corrected}).fetchone()
        if res:
            return res[0], True

    # 2. Try original name (ILIKE)
    query = sa.text("""
        SELECT DISTINCT m.station_name
        FROM meta_history AS m
        JOIN obs_raw AS o ON m.history_id = o.history_id
        WHERE m.station_name ILIKE :station_pattern
        LIMIT 1;
    """)
    with engine.connect() as conn:
        res = conn.execute(query, {"station_pattern": f"%{station_name}%"}).fetchone()
    if res:
        return res[0], False

    # 3. Try removing 'Wx' suffix (ILIKE)
    if station_name.endswith('Wx'):
        revised = station_name[:-2].strip()
        with engine.connect() as conn:
            res = conn.execute(query, {"station_pattern": f"%{revised}%"}).fetchone()
        if res:
            return res[0], False

    return None, None

def get_station_vars_from_db(engine, station_name):
    """Return variables and units from database, applying name corrections."""
    matched_name, is_exact = find_matching_station_in_db(engine, station_name)
    if matched_name is None:
        print(f"❌ No DB match for '{station_name}'")
        return None, None

    if is_exact:
        where_clause = "m.station_name = :station_pattern"
        pattern_param = matched_name
    else:
        where_clause = "m.station_name ILIKE :station_pattern"
        pattern_param = f"%{matched_name}%"

    query = sa.text(f"""
        SELECT DISTINCT v.net_var_name, v.unit
        FROM obs_raw AS o
        JOIN meta_history AS m ON o.history_id = m.history_id
        JOIN meta_vars AS v ON o.vars_id = v.vars_id
        WHERE {where_clause}
        ORDER BY v.net_var_name;
    """)
    with engine.connect() as conn:
        df = pd.read_sql(query, conn, params={"station_pattern": pattern_param})

    print(f"✅ '{station_name}' matched DB as '{matched_name}' ({'exact' if is_exact else 'LIKE'}) "
          f"→ {len(df)} vars found.")
    return df, matched_name

# The rest of your compare_file_vs_db function can remain unchanged.


# --- Step 3: Compare variables from Fern file vs DB ---
def compare_file_vs_db(df_station, csv_files, data_path, engine):
    results = []

    for station_i, csv_file in enumerate(csv_files):  # adjust range as needed
        station_file_name = os.path.splitext(csv_file)[0]
        df_data = pd.read_csv(os.path.join(data_path, csv_file), encoding='latin1', low_memory=False, nrows=100)
        station_name = df_station.iloc[station_i]['station_name']

        print(f"\n=== {station_name} ===")

        # Skip if mismatch
        if not names_match(station_name, station_file_name):
            print(f"⚠️ Station name mismatch: {station_name} vs {station_file_name}")
            continue

        # --- Fern file variables ---
        rows, native_id = create_rows_from_dataframe(df_station, df_data, station_i)
        rows_clean = [r for r in rows if not pd.isna(r.val)]
        variable_unit_fern = [(r.variable_name, r.unit) for r in rows_clean if hasattr(r, "variable_name")]
        var_to_unit_file = dict(set(variable_unit_fern))
        vars_from_file = set(var_to_unit_file.keys())

        # --- DB variables ---
        df_db, matched_name = get_station_vars_from_db(engine, station_name)
        if df_db is None:
            continue
        vars_from_db = set(df_db['net_var_name'])
        var_to_unit_db = dict(zip(df_db['net_var_name'], df_db['unit']))

        # --- Compare ---
        vars_in_both = sorted(vars_from_file & vars_from_db)
        vars_only_file = sorted(vars_from_file - vars_from_db)
        vars_only_db = sorted(vars_from_db - vars_from_file)



        print(f"✅ Common vars ({len(vars_in_both)}): {vars_in_both}")
        print(f"📄 Only in file ({len(vars_only_file)}): {vars_only_file}")
        print(f"🗄️ Only in DB ({len(vars_only_db)}): {vars_only_db}")

        # --- Save summary ---
        results.append({
            "station_name": station_name,
            "matched_db_name": matched_name,
            "vars_in_both": vars_in_both,
            "vars_only_file": vars_only_file,
            "vars_only_db": vars_only_db,
            "var_and_unit_db": var_to_unit_db

        })

    return results


# --- Example usage ---
results = compare_file_vs_db(df_station, csv_files, data_path, engine)

# Optional: inspect combined results as DataFrame
summary = pd.DataFrame(results)
summary.head()

summary.to_csv("fern_station_db_variable_compare_summary.csv", index=False)



=== Atlin School ===
✅ 'Atlin School' matched DB as 'Atlin' (exact) → 8 vars found.
✅ Common vars (0): []
📄 Only in file (11): ['Air Temp', 'Air Temp 2', 'Gd Temp 25 cm', 'Gd Temp 50 cm', 'Gd Temp 75 cm', 'Gust Speed', 'Rain', 'Sfc Temp', 'Solar Radiation', 'Wind Direction', 'Wind Speed']
🗄️ Only in DB (8): ['CURRENT_AIR_TEMPERATURE1', 'HEIGHT_OF_SNOW', 'MAXIMUM_AIR_TEMPERATURE', 'MINIMUM_AIR_TEMPERATURE', 'PRECIPITATION_NEW', 'RELATIVE_HUMIDITY1', 'STANDARD_SNOW', 'STORM_SNOW']

=== BarrenWx ===
✅ 'BarrenWx' matched DB as 'BarrenWx' (LIKE) → 13 vars found.
✅ Common vars (1): ['RH']
📄 Only in file (11): ['DewPt', 'Gust Speed', 'Pressure', 'Rain', 'Soil Temp', 'Solar Radiation', 'Temp', 'WC_cal', 'Water Content', 'Wind Direction', 'Wind Speed']
🗄️ Only in DB (12): ['DewPtC', 'GustSpeedms', 'Pressurembar', 'Rainmm', 'Soil_Temp', 'SolarRadiationWm', 'TempC', 'Water_Content_m3_m3_15_cm', 'Water_Content_m3_m3_30_cm', 'Water_Content_m3_m3_5_cm', 'WindDirection', 'WindSpeedms']

=== Blackhaw

In [10]:
### Cmpare variables, just print information

from rapidfuzz import process

def fuzzy_match_var(file_var, db_vars, threshold=85):
    # Get best match and score
    match, score, _ = process.extractOne(file_var, db_vars, scorer=fuzz.ratio)
    # Get all matches for additional info
    matches = process.extract(file_var, db_vars, limit=1, scorer=fuzz.ratio)
    # Keep original name if score below threshold
    result = match if score >= threshold else file_var
    return result, score, matches


# Compare variables for the first 3 stations in summary
for idx, row in summary.iterrows():
    station_name = row['station_name']
    matched_db_name = row['matched_db_name']
    db_vars = row['vars_in_both'] + row['vars_only_db']  # all DB vars for this station
    file_vars = row['vars_only_file']

    print(f"\n--- {station_name} ---")
    print(f"Variables in files: {file_vars}")
    print(f"Variables in db: {db_vars}")
    for file_var in file_vars:
        best_match, score, matches = fuzzy_match_var(file_var, db_vars, threshold=70)
        in_db = best_match in db_vars
        match_info = ' | '.join(f"{m}({s:.1f})" for m, s, _ in matches)
        print(f"{file_var:25s} -> {best_match:25s} | score={score:.1f} | in_db={in_db} | best: {match_info}")


--- Atlin School ---
Variables in files: ['Air Temp', 'Air Temp 2', 'Gd Temp 25 cm', 'Gd Temp 50 cm', 'Gd Temp 75 cm', 'Gust Speed', 'Rain', 'Sfc Temp', 'Solar Radiation', 'Wind Direction', 'Wind Speed']
Variables in db: ['CURRENT_AIR_TEMPERATURE1', 'HEIGHT_OF_SNOW', 'MAXIMUM_AIR_TEMPERATURE', 'MINIMUM_AIR_TEMPERATURE', 'PRECIPITATION_NEW', 'RELATIVE_HUMIDITY1', 'STANDARD_SNOW', 'STORM_SNOW']
Air Temp                  -> Air Temp                  | score=16.0 | in_db=False | best: PRECIPITATION_NEW(16.0)
Air Temp 2                -> Air Temp 2                | score=14.8 | in_db=False | best: PRECIPITATION_NEW(14.8)
Gd Temp 25 cm             -> Gd Temp 25 cm             | score=14.8 | in_db=False | best: HEIGHT_OF_SNOW(14.8)
Gd Temp 50 cm             -> Gd Temp 50 cm             | score=14.8 | in_db=False | best: HEIGHT_OF_SNOW(14.8)
Gd Temp 75 cm             -> Gd Temp 75 cm             | score=14.8 | in_db=False | best: HEIGHT_OF_SNOW(14.8)
Gust Speed                -> Gust Speed   

In [11]:
### Cmpare variables, save the matching results to a log file

import logging
from rapidfuzz import process

# Set up logging
log_path = '/workspaces/crmprtd/fern/all_stations_insert/'
log_file = "variable_matching_results.log"
match_logger = logging.getLogger("matching_logger")
match_logger.setLevel(logging.INFO)

# Clear existing handlers if any
if match_logger.hasHandlers():
    match_logger.handlers.clear()

# Add file handler
fh = logging.FileHandler(os.path.join(log_path, log_file), mode="w", encoding="utf-8")
fh.setLevel(logging.INFO)
formatter = logging.Formatter("%(message)s")
fh.setFormatter(formatter)
match_logger.addHandler(fh)

def fuzzy_match_var(file_var, db_vars, threshold=85):
    # Get best match and score
    match, score, _ = process.extractOne(file_var, db_vars, scorer=fuzz.ratio)
    # Get all matches for additional info
    matches = process.extract(file_var, db_vars, limit=1, scorer=fuzz.ratio)
    # Keep original name if score below threshold
    result = match if score >= threshold else file_var
    return result, score, matches

# Compare variables and log results
for idx, row in summary.iterrows():
    station_name = row['station_name']
    matched_db_name = row['matched_db_name']
    db_vars = row['vars_in_both'] + row['vars_only_db']
    file_vars = row['vars_only_file']

    header = f"\n--- {station_name} ---"
    print(header)
    match_logger.info(header)
    
    vars_info = f"Variables in files: {file_vars}"
    print(vars_info)
    match_logger.info(vars_info)
    
    db_vars_info = f"Variables in db: {db_vars}"
    print(db_vars_info)
    match_logger.info(db_vars_info)

    for file_var in file_vars:
        best_match, score, matches = fuzzy_match_var(file_var, db_vars, threshold=70)
        in_db = best_match in db_vars
        match_info = ' | '.join(f"{m}({s:.1f})" for m, s, _ in matches)
        result = f"{file_var:25s} -> {best_match:25s} | score={score:.1f} | in_db={in_db} | matches: {match_info}"
        print(result)
        match_logger.info(result)

match_logger.info("\n✅ Variable matching completed and logged.")

2025-11-03 21:31:28 - INFO - 
--- Atlin School ---
2025-11-03 21:31:28 - INFO - Variables in files: ['Air Temp', 'Air Temp 2', 'Gd Temp 25 cm', 'Gd Temp 50 cm', 'Gd Temp 75 cm', 'Gust Speed', 'Rain', 'Sfc Temp', 'Solar Radiation', 'Wind Direction', 'Wind Speed']
2025-11-03 21:31:28 - INFO - Variables in db: ['CURRENT_AIR_TEMPERATURE1', 'HEIGHT_OF_SNOW', 'MAXIMUM_AIR_TEMPERATURE', 'MINIMUM_AIR_TEMPERATURE', 'PRECIPITATION_NEW', 'RELATIVE_HUMIDITY1', 'STANDARD_SNOW', 'STORM_SNOW']
2025-11-03 21:31:28 - INFO - Air Temp                  -> Air Temp                  | score=16.0 | in_db=False | matches: PRECIPITATION_NEW(16.0)
2025-11-03 21:31:28 - INFO - Air Temp 2                -> Air Temp 2                | score=14.8 | in_db=False | matches: PRECIPITATION_NEW(14.8)
2025-11-03 21:31:28 - INFO - Gd Temp 25 cm             -> Gd Temp 25 cm             | score=14.8 | in_db=False | matches: HEIGHT_OF_SNOW(14.8)
2025-11-03 21:31:28 - INFO - Gd Temp 50 cm             -> Gd Temp 50 cm          

2025-11-03 21:31:28 - INFO - Gd Temp 75 cm             -> Gd Temp 75 cm             | score=14.8 | in_db=False | matches: HEIGHT_OF_SNOW(14.8)
2025-11-03 21:31:28 - INFO - Gust Speed                -> Gust Speed                | score=16.7 | in_db=False | matches: HEIGHT_OF_SNOW(16.7)
2025-11-03 21:31:28 - INFO - Rain                      -> Rain                      | score=14.3 | in_db=False | matches: STORM_SNOW(14.3)
2025-11-03 21:31:28 - INFO - Sfc Temp                  -> Sfc Temp                  | score=22.2 | in_db=False | matches: STORM_SNOW(22.2)
2025-11-03 21:31:28 - INFO - Solar Radiation           -> Solar Radiation           | score=16.0 | in_db=False | matches: STORM_SNOW(16.0)
2025-11-03 21:31:28 - INFO - Wind Direction            -> Wind Direction            | score=8.3 | in_db=False | matches: STORM_SNOW(8.3)
2025-11-03 21:31:28 - INFO - Wind Speed                -> Wind Speed                | score=10.0 | in_db=False | matches: STORM_SNOW(10.0)
2025-11-03 21:31:28 -


--- Atlin School ---
Variables in files: ['Air Temp', 'Air Temp 2', 'Gd Temp 25 cm', 'Gd Temp 50 cm', 'Gd Temp 75 cm', 'Gust Speed', 'Rain', 'Sfc Temp', 'Solar Radiation', 'Wind Direction', 'Wind Speed']
Variables in db: ['CURRENT_AIR_TEMPERATURE1', 'HEIGHT_OF_SNOW', 'MAXIMUM_AIR_TEMPERATURE', 'MINIMUM_AIR_TEMPERATURE', 'PRECIPITATION_NEW', 'RELATIVE_HUMIDITY1', 'STANDARD_SNOW', 'STORM_SNOW']
Air Temp                  -> Air Temp                  | score=16.0 | in_db=False | matches: PRECIPITATION_NEW(16.0)
Air Temp 2                -> Air Temp 2                | score=14.8 | in_db=False | matches: PRECIPITATION_NEW(14.8)
Gd Temp 25 cm             -> Gd Temp 25 cm             | score=14.8 | in_db=False | matches: HEIGHT_OF_SNOW(14.8)
Gd Temp 50 cm             -> Gd Temp 50 cm             | score=14.8 | in_db=False | matches: HEIGHT_OF_SNOW(14.8)
Gd Temp 75 cm             -> Gd Temp 75 cm             | score=14.8 | in_db=False | matches: HEIGHT_OF_SNOW(14.8)
Gust Speed                -

In [12]:
summary

,station_name,matched_db_name,vars_in_both,vars_only_file,vars_only_db,var_and_unit_db
0,Atlin School,Atlin,[],"[Air Temp, Air Temp 2, Gd Temp 25 cm, Gd Temp ...","[CURRENT_AIR_TEMPERATURE1, HEIGHT_OF_SNOW, MAX...","{'CURRENT_AIR_TEMPERATURE1': 'celsius', 'HEIGH..."
1,BarrenWx,BarrenWx,[RH],"[DewPt, Gust Speed, Pressure, Rain, Soil Temp,...","[DewPtC, GustSpeedms, Pressurembar, Rainmm, So...","{'DewPtC': 'celsius', 'GustSpeedms': 'm s-1', ..."
2,BlackhawkWx,Blackhawk,[RH],"[DewPt, Gust Speed, Pressure, Rain, Solar Radi...","[DewPtC, GustSpeedms, Pressurembar, Rainmm, Sn...","{'DewPtC': 'celsius', 'GustSpeedms': 'm s-1', ..."
3,BoulderWx,Boulder Creek,[RH],"[DewPt, Gust Speed, Pressure, Rain, Solar Radi...","[DewPtC, GustSpeedms, Pressurembar, Rainmm, So...","{'DewPtC': 'celsius', 'GustSpeedms': 'm s-1', ..."
4,BowronPit,Bowron Pit,[],"[DewPt 2, RH 2, Temp 2]","[DewPtC, GustSpeedms, Pressurembar, RH, Rainmm...","{'DewPtC': 'celsius', 'GustSpeedms': 'm s-1', ..."
5,BulkleyWx,BulkleyWx,[RH],"[DewPt, Gust Speed, Pressure, Rain, Solar Radi...","[DewPtC, GustSpeedms, Pressurembar, Rainmm, So...","{'DewPtC': 'celsius', 'GustSpeedms': 'm s-1', ..."
6,CPFWx,CPFWx,[RH],"[DewPt, Gust Speed, Pressure, Rain, Solar Radi...","[DewPtC, GustSpeedms, Pressurembar, Rainmm, So...","{'DewPtC': 'celsius', 'GustSpeedms': 'm s-1', ..."
7,Canoe Mountain Stn,Canoe Mountain Stn,[RH],"[DewPt, Gust Speed, Pressure, Rain, Solar Radi...","[DewPtC, GustSpeedms, Pressurembar, Rainmm, So...","{'DewPtC': 'celsius', 'GustSpeedms': 'm s-1', ..."
8,Caribou Pass,Caribou Pass,[RH],"[DewPt, DewPt 2, Gust Speed, Pressure, RH 2, R...","[DewPtC, GustSpeedms, Pressurembar, Rainmm, So...","{'DewPtC': 'celsius', 'GustSpeedms': 'm s-1', ..."
9,ChapmanWx,ChapmanWx,[RH],"[DewPt, Gust Speed, Pressure, Rain, Soil Temp,...","[DewPtC, GustSpeedms, Pressurembar, Rainmm, So...","{'DewPtC': 'celsius', 'GustSpeedms': 'm s-1', ..."


### For the varibles of perfect match, update the row objects' varible and unit to the one obtained in db.

In [13]:
def obtain_rows(df_station, data_path, csv_file, station_i, nrows=100):
    """
    Read a station CSV, create and clean observation rows.

    Parameters
    ----------
    df_station : pd.DataFrame
        DataFrame containing station metadata (e.g., name, id).
    data_path : str
        Path to folder containing CSV files.
    csv_file : str
        The specific CSV file name.
    station_i : int
        Index of the station in df_station.
    nrows : int
        Optional: number of rows to read (for testing/preview).

    Returns
    -------
    rows_clean : list
        List of cleaned observation rows.
    native_id : any
        Native ID returned by create_rows_from_dataframe().
    summary : dict
        Small summary containing station_name, total_vars, and variable counts.
    """
    station_name = df_station.iloc[station_i]['station_name']
    station_file_name = os.path.splitext(csv_file)[0]

    # --- Read file ---
    csv_path = os.path.join(data_path, csv_file)
    df_data = pd.read_csv(csv_path, encoding='latin1', low_memory=False, nrows=nrows)

    # --- Generate raw rows from the Fern file ---
    rows, native_id = create_rows_from_dataframe(df_station, df_data, station_i)

    # --- Clean rows (remove NaN values) ---
    rows_clean = [r for r in rows if hasattr(r, "val") and not pd.isna(r.val)]

    # --- Count variables ---
    var_counts = Counter(r.variable_name for r in rows_clean if hasattr(r, "variable_name"))
    total_var_counts = sum(var_counts.values())

    summary = {
        "station_name": station_name,
        "file_name": station_file_name,
        "total_var_counts": total_var_counts,
        "unique_variables": len(var_counts),
        "variable_counts": var_counts
    }

    return rows_clean, native_id, summary

for station_i, csv_file in enumerate(csv_files[0:3]):
    rows_clean, native_id, summary_row = obtain_rows(df_station, data_path, csv_file, station_i)

    print(f"\n=== {summary_row['station_name']} ===")
    print(f"📄 File: {summary_row['file_name']}")
    print(f"🔢 Total variables: {summary_row['unique_variables']}, Total counts: {summary_row['total_var_counts']}")
    print(f"🧩 Variable counts: {summary_row['variable_counts']}")


=== Atlin School ===
📄 File: Atlin school
🔢 Total variables: 11, Total counts: 1099
🧩 Variable counts: Counter({'Air Temp': 100, 'Wind Speed': 100, 'Gust Speed': 100, 'Wind Direction': 100, 'Solar Radiation': 100, 'Gd Temp 75 cm': 100, 'Gd Temp 50 cm': 100, 'Gd Temp 25 cm': 100, 'Sfc Temp': 100, 'Air Temp 2': 100, 'Rain': 99})

=== BarrenWx ===
📄 File: Barren
🔢 Total variables: 12, Total counts: 1399
🧩 Variable counts: Counter({'Water Content': 300, 'Pressure': 100, 'Temp': 100, 'RH': 100, 'DewPt': 100, 'Wind Speed': 100, 'Gust Speed': 100, 'Wind Direction': 100, 'Solar Radiation': 100, 'WC_cal': 100, 'Soil Temp': 100, 'Rain': 99})

=== BlackhawkWx ===
📄 File: Blackhawk
🔢 Total variables: 9, Total counts: 899
🧩 Variable counts: Counter({'Pressure': 100, 'Temp': 100, 'RH': 100, 'DewPt': 100, 'Wind Speed': 100, 'Gust Speed': 100, 'Wind Direction': 100, 'Solar Radiation': 100, 'Rain': 99})


In [14]:
def update_rows_with_db_vars(rows_clean, db_vars, db_vars_unit, threshold=85):
    """
    Update variable names and units in rows_clean using fuzzy matching to DB variables.
    Returns updated_rows, updated_vars, unchanged_vars.
    """
    updated_rows = []
    updated_vars = []
    unchanged_vars = []

    for r in rows_clean:
        best_match, score, matches = fuzzy_match_var(r.variable_name, db_vars, threshold=threshold)
        if score >= threshold and best_match in db_vars_unit:
            new_var = best_match
            new_unit = db_vars_unit[best_match]
            updated_vars.append((r.variable_name, new_var, r.unit, new_unit))
        else:
            new_var = r.variable_name
            new_unit = r.unit
            unchanged_vars.append((r.variable_name, r.unit))

        updated_rows.append(
            Row(
                time=r.time,
                val=r.val,
                variable_name=new_var,
                unit=new_unit,
                network_name=r.network_name,
                station_id=r.station_id,
                lat=r.lat,
                lon=r.lon
            )
        )
    return updated_rows, updated_vars, unchanged_vars


# --- Usage in your main loop ---
csv_path = "/workspaces/crmprtd/fern/all_stations_insert/fern_station_db_variable_compare_summary.csv"
var_compare_summary = pd.read_csv(csv_path)

for station_i, csv_file in enumerate(csv_files):
    rows_clean, native_id, summary_row = obtain_rows(df_station, data_path, csv_file, station_i, nrows=100)
    station_name = summary_row['station_name']

    var_station_line = var_compare_summary[var_compare_summary['station_name'] == station_name]
    if var_station_line.empty:
        print(f"Skipping {station_name}: not found in summary.")
        continue

    db_vars = eval(var_station_line.iloc[0]['vars_only_db']) if isinstance(var_station_line.iloc[0]['vars_only_db'], str) else var_station_line.iloc[0]['vars_only_db']
    db_vars_unit = eval(var_station_line.iloc[0]['var_and_unit_db']) if isinstance(var_station_line.iloc[0]['var_and_unit_db'], str) else var_station_line.iloc[0]['var_and_unit_db']

    updated_rows, updated_vars, unchanged_vars = update_rows_with_db_vars(rows_clean, db_vars, db_vars_unit, threshold=85)
    print(f"Station {station_name}: {len(updated_rows)} rows updated.")

Station Atlin School: 1099 rows updated.
Station BarrenWx: 1399 rows updated.
Station BlackhawkWx: 899 rows updated.
Station BoulderWx: 900 rows updated.
Station BowronPit: 300 rows updated.
Station BulkleyWx: 900 rows updated.
Station CPFWx: 900 rows updated.
Station Canoe Mountain Stn: 899 rows updated.
Station Caribou Pass: 1199 rows updated.
Station ChapmanWx: 1399 rows updated.
Station ChiefLakeWx: 1199 rows updated.
Station CoalmineWx: 300 rows updated.
Skipping CrookedLk: not found in summary.
Station CrystalWx: 1000 rows updated.
Station Dennis: 1299 rows updated.
Station DunsterWx: 899 rows updated.
Station EndakoWx: 1000 rows updated.
Station GeorgeWx: 1000 rows updated.
Station GunnelWx: 899 rows updated.
Station HourglassWx: 899 rows updated.
Station Hudson Bay Mtn2: 899 rows updated.
Skipping IBB2Wx: not found in summary.
Skipping IBB3Wx: not found in summary.
Skipping Inklin: not found in summary.
Station Kluskus: 899 rows updated.
Station MacJxnWx: 1000 rows updated.
Sta

In [40]:
len(db_vars)

13

In [42]:
db_vars = list(db_vars_unit.keys())


['DewPtC',
 'GustSpeedms',
 'Pressurembar',
 'Rainmm',
 'RH',
 'Soil_Temp',
 'SolarRadiationWm',
 'TempC',
 'Water_Content_m3_m3_15_cm',
 'Water_Content_m3_m3_30_cm',
 'Water_Content_m3_m3_5_cm',
 'WindDirection',
 'WindSpeedms']

In [65]:
def updated_rows_only_changed(rows_clean, db_vars_unit, threshold=60):
    """
    Return only rows where variable name was updated (not unchanged).
    """
    updated_rows_clean = []
    updated_vars = []

    updated_var_names = set()
    unchanged_var_names = set()

    db_vars = list(db_vars_unit.keys())

    print(db_vars)

    for r in rows_clean:
        best_match, score, matches = fuzzy_match_var(r.variable_name, db_vars, threshold=threshold)
        if score >= threshold and best_match in db_vars_unit:
            new_var = best_match
            new_unit = db_vars_unit[best_match]
            updated_vars.append((r.variable_name, new_var, r.unit, new_unit))
            updated_rows_clean.append(
                Row(
                    time=r.time,
                    val=r.val,
                    variable_name=new_var,
                    unit=new_unit,
                    network_name=r.network_name,
                    station_id=r.station_id,
                    lat=r.lat,
                    lon=r.lon
                )
            )
            updated_var_names.add(r.variable_name)
        
        elif score < threshold:
            unchanged_var_names.add(r.variable_name)

    print(f"\nStation {station_name}:")
    print(f"  {len(updated_var_names)} unique variables updated:")
    for var in sorted(updated_var_names):
        print(f"    UPDATED: {var}")

    print(f"  {len(unchanged_var_names)} unique variables unchanged:")
    for var in sorted(unchanged_var_names):
        print(f"    UNCHANGED: {var}")

    return updated_rows_clean, updated_var_names, unchanged_var_names



In [67]:
# --- Containers for all stations ---
all_updated_rows = []
all_updated_vars = []
all_unchanged_vars = []
all_total_rows = []  # store total rows per station
updated_total_rows = []  # store total rows per station

csv_path = "/workspaces/crmprtd/fern/all_stations_insert/fern_station_db_variable_compare_summary.csv"
var_compare_summary = pd.read_csv(csv_path)

threshold = 80

for station_i, csv_file in enumerate(csv_files):
    rows_clean, native_id, summary_row = obtain_rows(df_station, data_path, csv_file, station_i, nrows=10)
    station_name = summary_row['station_name']
    total_rows_count = len(rows_clean)
    all_total_rows.append({'station_name': station_name, 'total_rows': total_rows_count})

    var_station_line = var_compare_summary[var_compare_summary['station_name'] == station_name]
    if var_station_line.empty:
        print(f"Skipping {station_name}: not found in summary.")
        continue

    # Combine vars_in_both and vars_only_db for db_vars
    db_vars = []
    for col in ['vars_in_both', 'vars_only_db']:
        val = var_station_line.iloc[0][col]
        if isinstance(val, str):
            db_vars += eval(val)
        else:
            db_vars += val

    db_vars_unit = eval(var_station_line.iloc[0]['var_and_unit_db']) if isinstance(var_station_line.iloc[0]['var_and_unit_db'], str) else var_station_line.iloc[0]['var_and_unit_db']

    # print(rows_clean)

    updated_rows, updated_vars, unchanged_vars = updated_rows_only_changed(rows_clean, db_vars_unit, threshold=threshold)

    # --- Append to global containers ---
    all_updated_rows.extend(updated_rows)
    all_updated_vars.append({'station_name': station_name, 'updated_vars': updated_vars})
    all_unchanged_vars.append({'station_name': station_name, 'unchanged_vars': unchanged_vars})
    updated_rows_count = len(updated_rows)
    updated_total_rows.append({'station_name': station_name, 'updated_rows': updated_rows_count})


# --- Save all_updated_rows to a single pickle file ---
pickle_file_path = output_folder + "updated_rows_all_stations.pkl"
with open(pickle_file_path, "wb") as f:
    pickle.dump(all_updated_rows, f)

# --- Combine summary with counts ---
combined = []
for upd, unchg, total_row, updated_row in zip(all_updated_vars, all_unchanged_vars, all_total_rows, updated_total_rows):
    updated_list = upd["updated_vars"] if upd["updated_vars"] else []
    unchanged_list = unchg["unchanged_vars"] if unchg["unchanged_vars"] else []
    total_rows = total_row["total_rows"]
    updated_row = updated_row["updated_rows"]

    combined.append({
        "station_name": upd["station_name"],
        "updated_row": total_rows,
        "updated_rows_count": updated_row,   # number of variables updated
        "updated_vars": ", ".join(updated_list),
        "num_updated_vars": len(updated_list),
        "unchanged_vars": ", ".join(unchanged_list),
        "num_unchanged_vars": len(unchanged_list)
    })

df_summary = pd.DataFrame(combined)

# --- Save to CSV ---
df_summary.to_csv(output_folder + "updated_vars_summary_all_stations.csv", index=False)

print("✅ Summary CSV saved with total rows and updated counts per station.")

['CURRENT_AIR_TEMPERATURE1', 'HEIGHT_OF_SNOW', 'MAXIMUM_AIR_TEMPERATURE', 'MINIMUM_AIR_TEMPERATURE', 'PRECIPITATION_NEW', 'RELATIVE_HUMIDITY1', 'STANDARD_SNOW', 'STORM_SNOW']

Station Atlin School:
  0 unique variables updated:
  11 unique variables unchanged:
    UNCHANGED: Air Temp
    UNCHANGED: Air Temp 2
    UNCHANGED: Gd Temp 25 cm
    UNCHANGED: Gd Temp 50 cm
    UNCHANGED: Gd Temp 75 cm
    UNCHANGED: Gust Speed
    UNCHANGED: Rain
    UNCHANGED: Sfc Temp
    UNCHANGED: Solar Radiation
    UNCHANGED: Wind Direction
    UNCHANGED: Wind Speed
['DewPtC', 'GustSpeedms', 'Pressurembar', 'Rainmm', 'RH', 'Soil_Temp', 'SolarRadiationWm', 'TempC', 'Water_Content_m3_m3_15_cm', 'Water_Content_m3_m3_30_cm', 'Water_Content_m3_m3_5_cm', 'WindDirection', 'WindSpeedms']

Station BarrenWx:
  10 unique variables updated:
    UPDATED: DewPt
    UPDATED: Gust Speed
    UPDATED: Pressure
    UPDATED: RH
    UPDATED: Rain
    UPDATED: Soil Temp
    UPDATED: Solar Radiation
    UPDATED: Temp
    UPDA

In [22]:
db_vars

['RH',
 'DewPtC',
 'GustSpeedms',
 'Pressurembar',
 'Rainmm',
 'Soil_Temp',
 'SolarRadiationWm',
 'TempC',
 'Water_Content_m3_m3_15_cm',
 'Water_Content_m3_m3_30_cm',
 'Water_Content_m3_m3_5_cm',
 'WindDirection',
 'WindSpeedms']

In [21]:
db_vars_unit

{'DewPtC': 'celsius',
 'GustSpeedms': 'm s-1',
 'Pressurembar': 'millibar',
 'Rainmm': 'mm',
 'RH': '%',
 'Soil_Temp': 'celsius',
 'SolarRadiationWm': 'W m-2',
 'TempC': 'celsius',
 'Water_Content_m3_m3_15_cm': 'm3/m3',
 'Water_Content_m3_m3_30_cm': 'm3/m3',
 'Water_Content_m3_m3_5_cm': 'm3/m3',
 'WindDirection': 'degree',
 'WindSpeedms': 'm s-1'}

In [17]:
import pandas as pd

# Path to your file
csv_path = "/workspaces/crmprtd/fern/all_stations_insert/fern_station_db_variable_compare_summary.csv"

var_compare_summary = pd.read_csv(csv_path)

# Show the first few rows
# print(var_compare_summary.head())

var_station_line = var_compare_summary[var_compare_summary['station_name'] == summary_row['station_name']]
var_station_line['vars_only_file']

db_vars = var_station_line['vars_only_db']  # all DB vars for this station
file_vars = var_station_line['vars_only_file']
db_vars_unit = var_station_line['var_and_unit_db']  # all DB vars for this station

for file_var in file_vars:
    best_match, score, matches = fuzzy_match_var(file_var, db_vars, threshold=70)

    print(file_var, best_match)


['DewPt', 'Gust Speed', 'Pressure', 'Rain', 'Soil Temp', 'Solar Radiation', 'Temp', 'WC_cal', 'Water Content', 'Wind Direction', 'Wind Speed'] ['DewPtC', 'GustSpeedms', 'Pressurembar', 'Rainmm', 'Soil_Temp', 'SolarRadiationWm', 'TempC', 'Water_Content_m3_m3_15_cm', 'Water_Content_m3_m3_30_cm', 'Water_Content_m3_m3_5_cm', 'WindDirection', 'WindSpeedms']
